In [ ]:
# ! pip install numba
from numba import njit
import numpy as np

In [ ]:
wind_speed = 10.0  # m/s RANGE ~ 0.1-100
fluid_density = 1.2 # kg/m^3 RANGE ~ 1.2 @ 20°C - 1.3 @ cold or mountains for air, 998 or ~1000 for water
fluid_kinematic_viscosity = 1.5e-5 # m^2/s RANGE ~ 1.5e-5 for air @ 20°C, 1e-6 for water
cell_size = 0.02 # m RANGE ~ 0.001 - 0.5 RULE: Whole body >= 40 cells, smallest important feature >= 4 cells 
# virtuell wind tunnel dimensions
domain_length = 10.0 # m IDEAL 3-5 length before and ~10 after the object
domain_width = 3.0 # m model frontal area / tunnel cross section under 5%
domain_height = 3.0 # m model frontal area / tunnel cross section under 5%

characteristic_length = 1.8

# Lattice Mach number = lattice_speed / sqrt(1/3) = 0.087 KEEP Mach under 0.1 max. 0.3
lattice_speed = 0.05 # RANGE 0.01 to 0.1 SMALLER = more accurate; #Steps ~ 1/lattice_speed

smagorinsky_constant = 0.17 # RANGE 0.1-0.24; Lower = less artificial damping but less stable.as_integer_ratio
# One flow-through = air travels through the whole tunnel once.
flow_throughs_to_simulate = 6 # A few to wash out the start up last to average RANGE 5-10

direction_arrows = np.array([
    # Rest vector
    [0, 0, 0],

    # Six axis directions
    [ 1, 0, 0],
    [-1, 0, 0],
    [ 0, 1, 0],
    [ 0,-1, 0],
    [ 0, 0, 1],
    [ 0, 0,-1],

    # Twelve diagonal directions
    [ 1, 1, 0],
    [-1, 1, 0],
    [ 1,-1, 0],
    [-1,-1, 0],

    [ 1, 0, 1],
    [-1, 0, 1],
    [ 1, 0,-1],
    [-1, 0,-1],

    [ 0, 1, 1],
    [ 0,-1, 1],
    [ 0, 1,-1],
    [ 0,-1,-1]
], dtype=np.int8)
direction_weights = np.array([
    1/3,

    # Six axis directions
    1/18, 1/18, 1/18,
    1/18, 1/18, 1/18,

    # Twelve diagonal directions
    1/36, 1/36, 1/36, 1/36,
    1/36, 1/36, 1/36, 1/36,
    1/36, 1/36, 1/36, 1/36
], dtype=np.float32)
opposite_directions = np.array([
    np.where((direction_arrows == -arrow).all(axis=1))[0][0]
    for arrow in direction_arrows])

In [ ]:
def derive_values(lattice_speed, cell_size, wind_speed, characteristic_length, fluid_kinematic_viscosity, domain_length, domain_width, domain_height, flow_throughs_to_simulate):

    time_step = lattice_speed * cell_size / wind_speed # seconds per loop
    lattice_viscosity = fluid_kinematic_viscosity * time_step / (cell_size * cell_size) # lattice units
    speed_of_sound_squared = 1.0 / 3.0 # D3Q19 fixed value
    relaxation_time = lattice_viscosity / speed_of_sound_squared + 0.5 # Chapman-Enskog expansion, must be > 0.5 for stability
    reynolds_number = wind_speed * characteristic_length / fluid_kinematic_viscosity # dimensionless, for reference only
    steps_per_flow_through = round(domain_length / cell_size) / lattice_speed # dimensionless, for reference only
    total_steps = int(steps_per_flow_through * flow_throughs_to_simulate) # dimensionless, for reference only
    
    nx = round(domain_width / cell_size)
    ny = round(domain_height / cell_size)
    nz = round(domain_length / cell_size)
    is_solid = np.zeros((nx, ny, nz), dtype=bool, )
    air_amounts = np.zeros((19, nx, ny, nz), dtype=np.float32)

    return time_step, lattice_viscosity, speed_of_sound_squared, relaxation_time, reynolds_number, steps_per_flow_through, total_steps, direction_arrows, direction_weights, is_solid, air_amounts, nx, ny, nz